In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns 

#### This time, I plan to use TF-IDF combined with embeddings (using a pre-trained FastText model) to further process the text in order to capture more information and improve model performance.

# Data Import and Clean

In [ ]:
url = "/kaggle/input/nlp-getting-started/train.csv"
data = pd.read_csv(url, encoding="ISO-8859-1")
data.head()

In [ ]:
data.shape

#### Let's clean the data first

In [ ]:
import re
import unicodedata

def clean_text(text):
    # Ensure input is a string
    text = str(text)
    
    # 1. Remove non-standard Unicode characters (commonly causes garbled text)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8", "ignore")
    
    # 2. Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # 3. Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)

    # 4. Remove non-alphabetic characters (optional)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # 5. Convert text to lowercase
    text = text.lower()
    
    # 6. Remove extra whitespaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

data["clean_text"] = data["text"].apply(clean_text)
data["clean_text"]

#### Import a SentenceTransformer pre-trained model.

In [ ]:
from sentence_transformers import SentenceTransformer

path = "/kaggle/input/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2"
model = SentenceTransformer(path)

In [ ]:
sentences = data["clean_text"].tolist()
sentence_embeddings = model.encode(sentences)

print(sentence_embeddings.shape)
print(sentence_embeddings[0].shape)

In [ ]:
embeddings_text = pd.DataFrame(sentence_embeddings, 
                             columns=[f"embed_{i}" for i in range(sentence_embeddings.shape[1])])
embeddings_text

# Model Training

#### Train test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(embeddings_text, data["target"], test_size=0.2, random_state=69)  

#### Using the function in the last notebook for NLP.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import time

def model_experiment(grid_para, classifier_ins, df):
    """
    Performs experiments for regression models, including feature selection, preprocessing, grid search, and performance evaluation.
    
    Args:
        grid_para (dict): A dictionary containing parameters for grid search.
        regressor_ins (sklearn.base.RegressorMixin): An instance of a regression model (e.g., LinearRegression(), XGBRegressor()).
        df_results (pd.DataFrame): The DataFrame used to store results.
    
    Returns:
        pd.DataFrame: The updated results DataFrame.
    """
    #use the parameters 
    grid_para = grid_para
    #create k-fold 
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=43)
    
    #create grid search
    grid_search = GridSearchCV(estimator=classifier_ins,
                           param_grid=grid_para,
                           cv=kfold,
                           scoring="accuracy",
                           n_jobs=-1,
                           verbose=1)
    
    #find the best model and count time
    start_time = time.time()

    
    # start model
    grid_search.fit(X_train, y_train)

    # grid_search.fit(X_train, y_train)
    best_model=grid_search.best_estimator_
    
    #print the score
    print("\nGridSearchCV execution completed.")
    print(f"Best hyperparameters: {grid_search.best_params_}")
    print("--------------------------------------------------------------------------")
    print(f"Train Accuracy: {best_model.score(X_train, y_train):}")
    print(f"Average accuracy of the best model during cross-validation: {grid_search.best_score_:}")
    print(f"Final Test Accuracy = {best_model.score(X_test, y_test):}")

    end_time = time.time()
    training_time = round(end_time - start_time, 2)
    
    #put in the dataframe
    new_row={        
        "Best hyperparameters":grid_search.best_params_, 
        "Train Accuracy": round(best_model.score(X_train, y_train), 4),
        "Cross-validation Accuracy" : round(grid_search.best_score_ ,4),
        "Final Test Accuracy": round(best_model.score(X_test, y_test), 4),
        "Best Model": best_model,
        "Training Time":training_time
        }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    #draw the confusion matrix
    
    y_predict=best_model.predict(X_test)
    
    cm = confusion_matrix(y_test, y_predict)
    ax= plt.subplot()
    sns.heatmap(cm, annot=True, ax = ax, fmt="d"); 
    ax.set_xlabel("Predicted labels")
    ax.set_ylabel("True labels")
    ax.set_title("Confusion Matrix"); 
    ax.xaxis.set_ticklabels(["Not Disaster", "Disaster"]); ax.yaxis.set_ticklabels(["Not Disaster", "Disaster"]) 
    plt.show() 

    return df

#### Initialize the classfier

In [ ]:
import xgboost as xgb

xgb_results = pd.DataFrame(columns=[        
    "Best hyperparameters",     
    "Train Accuracy",
    "Cross-validation Accuracy",         
    "Final Test Accuracy", 
    "Best Model",
    "Training Time"
])
xgb_class = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="logloss")

In [ ]:
grid_para= {
    "n_estimators": [50, 80], 
    "learning_rate": [0.05, 0.1], 
    "max_depth": [1, 3, 5], 
    "subsample": [0.6, 0.7], 
    "colsample_bytree": [0.6, 0.7], 
    "reg_alpha": [0, 0.5], 
    "reg_lambda": [1, 3] 
}

xgb_results = model_experiment(grid_para, xgb_class, xgb_results)
xgb_results

In [ ]:
grid_para= {
    "n_estimators": [50, 80], 
    "learning_rate": [0.01, 0.03], 
    "max_depth": [1, 2, 3], 
    "subsample": [0.6, 0.7], 
    "colsample_bytree": [0.6, 0.7], 
    "reg_alpha": [0, 0.5], 
    "reg_lambda": [1, 3] 
}

xgb_results = model_experiment(grid_para, xgb_class, xgb_results)
xgb_results

#### Utilize the early-stopping to reduce overfitting.

In [ ]:
best_params = xgb_results.iloc[-2, 0]

# Set the best paramater in the best model.
final_model = xgb.XGBClassifier(
    **best_params,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss",
    early_stopping_rounds=10
    )

X_train_sub, X_valid, y_train_sub, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
    )

#Fit the model
final_model.fit(
    X_train_sub, y_train_sub,
    eval_set=[(X_valid, y_valid)],
    verbose=False
    )

# Evaluation
train_acc = final_model.score(X_train, y_train)
test_acc = final_model.score(X_test, y_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Final Test Accuracy = {test_acc:.4f}")

# Submit

In [ ]:
url2 = "/kaggle/input/nlp-getting-started/sample_submission.csv"
sample_submission = pd.read_csv(url2)
sample_submission.head()

In [ ]:
url3 = "/kaggle/input/nlp-getting-started/test.csv"
test_data = pd.read_csv(url3)
test_data.head()

In [ ]:
test_data["clean_text"] = test_data["text"].apply(clean_text).apply(lambda text : re.sub(r'\b(a)\1{1,}\b', '', text))

test_sentences = test_data["clean_text"].tolist()
test_sentence_embeddings = model.encode(test_sentences)

embeddings_test_text = pd.DataFrame(test_sentence_embeddings, 
                             columns=[f"embed_{i}" for i in range(test_sentence_embeddings.shape[1])])
embeddings_test_text

In [ ]:
sample_submission["target"] = final_model.predict(embeddings_test_text)
sample_submission.head()

In [ ]:
sample_submission.to_csv("submission.csv", index=False)

# Conclusion

#### In a data cleaning scenario similar to my previous attempt, and under comparable machine learning hyperparameter configurations, I switched to using SentenceTransformer for NLP processing. 
#### This resulted in an improved prediction performance compared to before. It's clear that leveraging sentence context as vectors is more beneficial for this competition. Moving forward, 
#### I plan to learn more complex NLP methods that require deeper background knowledge to further enhance my skills..